# B-Free x GlobalForge — Notebook 02: Kaggle Training (K2, LoRA r=16)

**Target B: RTX PRO 6000 Blackwell 96GB, hoàn toàn OFFLINE.** Huấn luyện **DINOv2 ViT-B/14 reg4 + LIB + GSR + L_DCS** (LoRA r=16, α=32) trên B-Free training data (51.517 real + 309.102 fake), 504×504, 8 epochs, bf16.

**Inputs (attach trước khi chạy):**
1. **Output notebook 01** (wheels + bfree_src + DINOv2 weights + manifest) — qua *Add Input → Your Work → notebook 01*
2. **B-Free training data** (upload một lần từ grip.unina.it): dataset chứa `COCO_real_512/` + 6 thư mục `SD2.1_*/` (fake variants dùng cùng tên file với real)

**Locked decisions (plan.md D1–D10):** D1 lambda_dcs=0.01, tau=0.07, ls=0.1 · D2 8 epochs @504px · D3 full data · D4 50/50 per-ID pairing (p=0.5 real else random 1/6 fake) · D5 DINOv2 pretrained init offline · D6 auto batch size (0.9×VRAM) · D7 bf16 autocast · AdamW lr=1e-4 wd=1e-4, cosine eta_min=1e-7 per-step, clip 1.0 · val split `md5(id)%100<3`, mỗi ID = 1 real + 1 deterministic fake (balanced).

> **Vì sao không gọi `train_lora.py` trực tiếp?** Script của repo (a) yêu cầu CSV tĩnh `filename,label` — không thực hiện được pairing động D4 (resample mỗi epoch); (b) loop của nó **fp32**, vi phạm D7 (bf16 autocast); (c) không có đường load pretrained offline D5 (resample pos_embed 518→504); (d) checkpoint của nó thiếu CONFIG + train_log mà K3 cần. Notebook này **tái sử dụng repo như thư viện** — `BFreeGlobalForgeViT`, `DegradationPipeline`, `apply_lora_to_backbone`, `dmetrics` — đúng tinh thần hợp đồng `MODULES_INTERFACE.md`, chỉ thay phần orchestration cho khớp D4/D5/D6/D7.

In [ ]:
import logging
import os
import sys
from pathlib import Path

logger = logging.getLogger("bfree")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_console = logging.StreamHandler()
_console.setFormatter(logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S"))
logger.addHandler(_console)

logger.info("stdlib + logging ready")

In [ ]:
import glob
from pathlib import Path

WORK = Path("/kaggle/working")


def find_one(patterns, what, optional=False):
    """Return the first existing match across glob patterns (in priority order)."""
    for pat in patterns:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    if optional:
        return None
    raise FileNotFoundError(
        f"NOT FOUND: {what}\nsearched: {list(patterns)}\n"
        "Attach the missing Kaggle dataset / notebook-output input and re-run.")


def find_dir_containing(marker_rel, patterns, what, optional=False):
    """Return the parent dir that contains marker_rel, searched via glob patterns."""
    for pat in patterns:
        for hit in sorted(glob.glob(pat)):
            root = Path(hit)
            while root != Path("/kaggle/input") and root != root.parent:
                if (root / marker_rel).is_file():
                    return root
                root = root.parent
    if optional:
        return None
    raise FileNotFoundError(
        f"NOT FOUND: {what} (marker: {marker_rel})\nsearched: {list(patterns)}\n"
        "Attach the missing Kaggle dataset / notebook-output input and re-run.")


# ---- bundle produced by notebook 01 (attach its output as input here) ----
WHEELS_DIR = find_one(
    ["/kaggle/input/*/wheels_rtxpro6000",
     "/kaggle/input/wheels_rtxpro6000",
     "/kaggle/input/*/wheels*",
     "/kaggle/input/wheels*"],
    "wheel bundle 'wheels_rtxpro6000' (output of notebook 01)")

REPO_MARKER = "code/networks/bfree_globalforge_vit.py"
REPO_SRC = find_dir_containing(
    REPO_MARKER,
    ["/kaggle/input/*/bfree_src/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/bfree_src*/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/*/B-Free/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/B-Free*/code/networks/bfree_globalforge_vit.py",
     "/kaggle/input/*/code/networks/bfree_globalforge_vit.py"],
    "B-Free repo source 'bfree_src' (output of notebook 01)")

DINOV2_SD = find_one(
    ["/kaggle/input/*/models/vit_base_patch14_reg4_dinov2/model.safetensors",
     "/kaggle/input/models/vit_base_patch14_reg4_dinov2/model.safetensors",
     "/kaggle/input/dinov2*/*.safetensors",
     "/kaggle/input/*/dinov2*/*.safetensors"],
    "DINOv2 ViT-B/14 reg4 weights (output of notebook 01)")

wheels = sorted(glob.glob(str(Path(WHEELS_DIR) / "*.whl")))
assert wheels, f"No .whl files inside {WHEELS_DIR}"
logger.info(f"WHEELS_DIR  = {WHEELS_DIR} ({len(wheels)} wheels)")
logger.info(f"REPO_SRC    = {REPO_SRC}")
logger.info(f"DINOV2_SD   = {DINOV2_SD}")

# ---- external input: B-Free training data (COCO_real_512 + SD2.1_*) ----
TRAIN_DATA_ROOT = None
for cand in sorted(glob.glob("/kaggle/input/*/")) + ["/kaggle/input/"]:
    for rel in ("COCO_real_512", "bfree-training-data/COCO_real_512"):
        if Path(cand, rel).is_dir():
            TRAIN_DATA_ROOT = Path(cand, rel).parent
            break
    if TRAIN_DATA_ROOT is not None:
        break
if TRAIN_DATA_ROOT is None:
    raise FileNotFoundError(
        "B-Free training data not found: no /kaggle/input/*/COCO_real_512. "
        "Upload it (grip.unina.it training_data) as a Kaggle dataset and attach.")
logger.info(f"TRAIN_DATA_ROOT = {TRAIN_DATA_ROOT}")

OUTPUTS = WORK
OUTPUTS.mkdir(parents=True, exist_ok=True)
_fh = logging.FileHandler(str(OUTPUTS / "train_log.txt"), mode="a", encoding="utf-8")
_fh.setFormatter(logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(_fh)

In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

logger.info("offline env vars set (HF hub blocked, bytecode off — repo source stays read-only in input)")

In [ ]:
!pip install --no-index --find-links="{WHEELS_DIR}" \
    torch torchvision timm peft transformers accelerate \
    pandas numpy matplotlib seaborn scikit-learn scipy \
    pyyaml pillow tqdm safetensors huggingface_hub

import torch
import timm
import peft
import transformers

assert torch.__version__.startswith("2.8.0"), f"torch {torch.__version__} != 2.8.0+cu128 (bundle wrong?)"
assert transformers.__version__ == "4.55.4", f"transformers {transformers.__version__} != 4.55.4"
assert peft.__version__ == "0.15.2", f"peft {peft.__version__} != 0.15.2"
import pandas
assert pandas.__version__.split(".")[0] == "2", "pandas>=3 breaks sklearn (risk table)"
logger.info(f"pip --no-index OK | torch={torch.__version__} timm={timm.__version__} "
            f"peft={peft.__version__} transformers={transformers.__version__}")

In [ ]:
import sys

sys.path.insert(0, str(Path(REPO_SRC) / "code"))

stubs = sorted(glob.glob(str(Path(REPO_SRC) / "code" / "modules" / "*_stub.py")))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT
from configs.loader import load_config, build_model_kwargs
from datasets.bfree_dataset import BFreeDataset, DegradationPipeline
from modules.lib_adapter import LIBAdapter
from modules.gsr_adapter import GSRAdapter
from modules.dcs_loss import DCSLoss, info_nce_loss
import networks.bfree_globalforge_vit as _bgv

logger.info(f"repo import OK from {_bgv.__file__} (no stubs — K0 verified)")

In [ ]:
# ============================================================
# Hyperparameters (locked D1-D10) - everything lives here
# ============================================================
import random

import numpy as np

CONFIG = {
    "arch": "vit_base_patch14_reg4_dinov2.lvd142m",
    "num_classes": 2,
    "img_size": 504,
    "lambda_dcs": 0.01,            # D1
    "dcs_tau": 0.07,
    "label_smoothing": 0.1,
    "lib_kernel": 3,
    "lib_tau": 0.5,
    "gsr_window": 3,
    "gsr_mask_prob": 1.0,
    "epochs": 8,                   # D2
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_targets": ["qkv", "proj", "fc1", "fc2"],   # GlobalForge convention
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "max_grad_norm": 1.0,
    "scheduler_eta_min": 1e-7,
    "max_vram_frac": 0.9,           # D6
    "batch_hard_cap": 128,
    "val_md5_percentile": 3,        # md5(id)%100 < 3
    "val_fake_variant": "SD2.1_selfconditioned",
    "pair_real_prob": 0.5,         # D4
    "num_workers": 4,
    "seed": 42,
}
DEVICE = "cuda:0"

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

REAL_DIR = str(TRAIN_DATA_ROOT / "COCO_real_512")
FAKE_DIRS = sorted(str(p) for p in TRAIN_DATA_ROOT.glob("SD2.1_*") if p.is_dir())
assert Path(REAL_DIR).is_dir(), f"COCO_real_512 missing under {TRAIN_DATA_ROOT}"
assert len(FAKE_DIRS) == 6, f"Expected 6 SD2.1_* dirs, found {len(FAKE_DIRS)}"
for d in FAKE_DIRS:
    logger.info(f"  {Path(d).name}: {len(list(Path(d).glob('*')))} files")
logger.info(f"CONFIG: {CONFIG}")

### Dataset — 50/50 per-ID pairing (D4)

`BFreeDataset` của repo nhận CSV tĩnh nên không pairing động theo ID được; cell dưới tái dùng `DegradationPipeline` của repo và cài đúng convention D4:

- **Train**: 1 sample = 1 ID; `p=0.5` → real, ngược lại random 1/6 fake variant (resample mỗi `__getitem__` → 8 epochs phủ hết các variant).
- **Val** (`md5(stem)%100 < 3`): mỗi ID 2 samples — chẵn = real, lẻ = deterministic fake `SD2.1_selfconditioned` → balanced.
- Crop: train RandomCrop 504 (ảnh 512×512) + hflip 0.5; val center crop 504.
- Degraded view cho L_DCS: JPEG 20-80 → GaussianBlur k7 σ0.5-1.5 p0.8 → ColorJitter p0.8 (pipeline GlobalForge).

In [ ]:
import hashlib

from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from utils.normalization import get_list_norm


def build_stem_index(dirs):
    maps = []
    for d in dirs:
        m = {}
        for p in Path(d).glob("*"):
            if p.is_file():
                m[p.stem] = str(p)
        maps.append(m)
    return maps

REAL_MAP = build_stem_index([REAL_DIR])[0]
FAKE_MAPS = build_stem_index(FAKE_DIRS)
ALL_STEMS = sorted(REAL_MAP.keys())
logger.info(f"real={len(REAL_MAP)}, fake variants={[len(m) for m in FAKE_MAPS]}")


def is_val_id(stem):
    return int(hashlib.md5(stem.encode()).hexdigest(), 16) % 100 < CONFIG["val_md5_percentile"]

TRAIN_IDS = [s for s in ALL_STEMS if not is_val_id(s)]
VAL_IDS = [s for s in ALL_STEMS if is_val_id(s)]
logger.info(f"train IDs={len(TRAIN_IDS)}, val IDs={len(VAL_IDS)} (~{CONFIG['val_md5_percentile']}%)")


class PairDataset(Dataset):
    '''Train: p=0.5 real else random 1/6 fake variant (resample moi epoch).
    Val: 2 samples/ID - chan=real, le=deterministic fake variant (balanced).'''

    def __init__(self, ids, img_size, is_train, pair_real_prob, val_fake_variant):
        self.ids = ids
        self.img_size = img_size
        self.is_train = is_train
        self.pair_real_prob = pair_real_prob
        self.val_fake_variant = val_fake_variant
        self.normalize = T.Compose(get_list_norm("resnet"))
        self.degradation = DegradationPipeline()

    def __len__(self):
        return len(self.ids) * (1 if self.is_train else 2)

    def _pick_val_fake(self, stem):
        for m, d in zip(FAKE_MAPS, FAKE_DIRS):
            if Path(d).name == self.val_fake_variant and stem in m:
                return m[stem]
        for m in FAKE_MAPS:
            if stem in m:
                return m[stem]
        return None

    def __getitem__(self, i):
        stem = self.ids[i if self.is_train else i // 2]
        if self.is_train:
            if random.random() < self.pair_real_prob or not any(stem in m for m in FAKE_MAPS):
                path, label = REAL_MAP[stem], 0
            else:
                maps = [m for m in FAKE_MAPS if stem in m]
                path = maps[random.randrange(len(maps))][stem]
                label = 1
        else:
            if i % 2 == 0:
                path, label = REAL_MAP[stem], 0
            else:
                path = self._pick_val_fake(stem)
                label = 1
        assert path is not None, f"No image found for id {stem}"
        img = Image.open(path).convert("RGB")

        if self.is_train:
            t, l, h, w = T.RandomCrop.get_params(img, (self.img_size, self.img_size))
            img = TF.crop(img, t, l, h, w)
            if random.random() > 0.5:
                img = TF.hflip(img)
        else:
            img = TF.center_crop(img, (self.img_size, self.img_size))

        img_deg = self.degradation(img)
        return self.normalize(img), self.normalize(img_deg), label

train_ds = PairDataset(TRAIN_IDS, CONFIG["img_size"], True, CONFIG["pair_real_prob"], CONFIG["val_fake_variant"])
val_ds = PairDataset(VAL_IDS, CONFIG["img_size"], False, CONFIG["pair_real_prob"], CONFIG["val_fake_variant"])

a, b, c = train_ds[0]
_, _, c2 = val_ds[1]
assert tuple(a.shape) == (3, CONFIG["img_size"], CONFIG["img_size"])
logger.info(f"train_ds={len(train_ds)}, val_ds={len(val_ds)} | labels val[0],val[1]={c},{c2} (expect 0,1)")

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws)
    random.seed(ws)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

### Model — BFreeGlobalForgeViT + LoRA r=16 (offline pretrained init, D5)

Load `model.safetensors` từ bundle notebook 01 vào backbone; **resample `pos_embed` 518px (37×37) → 504px (36×36)** vì checkpoint DINOv2 mặc định 518px; filter key+shape mismatch (head 2-class mới sẽ missing — đúng). Sau đó áp LoRA r=16 α=32 qua `apply_lora_to_backbone` của repo (target `qkv/proj/fc1/fc2`; LIB/GSR/fc_norm/head vẫn full-train).

In [ ]:
import math

import torch


def load_pretrained_backbone(model, safetensors_path):
    '''D5 offline: load DINOv2 weights vao model.model + resample pos_embed 518->504.'''
    from safetensors.torch import load_file

    sd = load_file(str(safetensors_path))
    sd = {k[len("model."):] if k.startswith("model.") else k: v for k, v in sd.items()}

    prefix = model.model.num_prefix_tokens
    if "pos_embed" in sd:
        pe = sd["pos_embed"]
        old_hw = int(math.isqrt(pe.shape[1] - prefix))
        new_hw = model.model.patch_embed.grid_size[0]
        if old_hw != new_hw:
            from timm.layers.pos_embed import resample_abs_pos_embed
            logger.info(f"resample pos_embed: {old_hw}x{old_hw} -> {new_hw}x{new_hw}")
            sd["pos_embed"] = resample_abs_pos_embed(
                pe, new_size=model.model.patch_embed.grid_size,
                old_size=(old_hw, old_hw), num_prefix_tokens=prefix)

    ref = model.model.state_dict()
    dropped = [k for k, v in sd.items() if k not in ref or ref[k].shape != v.shape]
    sd = {k: v for k, v in sd.items() if k not in dropped}

    report = model.model.load_state_dict(sd, strict=False)
    logger.info(f"pretrained load: dropped={len(dropped)}, missing={report.missing_keys}")
    assert set(report.missing_keys) <= {"head.weight", "head.bias"}, "backbone not fully initialized"
    assert not report.unexpected_keys
    return model

from train_lora import apply_lora_to_backbone

model = BFreeGlobalForgeViT(
    arch=CONFIG["arch"], num_classes=CONFIG["num_classes"],
    img_size=CONFIG["img_size"], pretrained=False,
    use_lib=True, use_gsr=True, use_dcs=True,
    lib_kernel=CONFIG["lib_kernel"], lib_tau=CONFIG["lib_tau"],
    gsr_window=CONFIG["gsr_window"], gsr_mask_prob=CONFIG["gsr_mask_prob"],
    dcs_tau=CONFIG["dcs_tau"], lambda_dcs=CONFIG["lambda_dcs"],
    label_smoothing=CONFIG["label_smoothing"],
)
model = load_pretrained_backbone(model, DINOV2_SD)
model = apply_lora_to_backbone(model, r=CONFIG["lora_rank"],
                               lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"])
model.to(DEVICE)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
logger.info(f"trainable params: {n_train:,} / {n_total:,} ({100 * n_train / n_total:.2f}%)")

### Auto batch size finder (D6)

Doubling từ 2 → OOM/hard-cap, rồi binary search — đo bằng chính `compute_loss` (2 forwards clean+degraded, bf16, backward).

In [ ]:
def try_batch(model, bs, img_size, device):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        x1 = torch.randn(bs, 3, img_size, img_size, device=device)
        x2 = torch.randn(bs, 3, img_size, img_size, device=device)
        y = torch.randint(0, 2, (bs,), device=device)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            total, _, _ = model.compute_loss(x1, x2, y)
        total.backward()
        del x1, x2, y, total
        return True
    except torch.cuda.OutOfMemoryError:
        return False
    finally:
        torch.cuda.empty_cache()

def find_max_batch_size(model, img_size, device, hard_cap):
    assert try_batch(model, 2, img_size, device), "Batch size 2 OOMs — environment problem."
    lo, bs, hi = 2, 4, None
    while bs <= hard_cap:
        if try_batch(model, bs, img_size, device):
            lo = bs
            bs *= 2
        else:
            hi = bs
            break
    if hi is None:
        logger.warning(f"no OOM up to hard cap {hard_cap}")
        hi = lo + 1
    while lo + 1 < hi:
        mid = (lo + hi) // 2
        if try_batch(model, mid, img_size, device):
            lo = mid
        else:
            hi = mid
    return lo

BATCH_SIZE = find_max_batch_size(model, CONFIG["img_size"], DEVICE, CONFIG["batch_hard_cap"])
CONFIG["batch_size"] = BATCH_SIZE
logger.info(f"AUTO BATCH SIZE = {BATCH_SIZE}")

### Training loop — 8 epochs, bf16 autocast (D7), clip 1.0

`total = CE + lambda_dcs * DCS` qua `model.compute_loss`; **log riêng CE và DCS** (rule agent.md) → `train_log.csv`; AdamW lr 1e-4 wd 1e-4; CosineAnnealingLR `T_max=epochs*len(train_loader)`, eta_min=1e-7, step mỗi batch; lưu checkpoint best theo val_bAcc + checkpoint cuối (kèm CONFIG — K3 cần).

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from utils.dmetrics import balanced_accuracy_score, roc_auc_score

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True,
                           persistent_workers=True, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_ds, batch_size=max(2, BATCH_SIZE // 2), shuffle=False,
                        num_workers=CONFIG["num_workers"], pin_memory=True)

optimizer = AdamW((p for p in model.parameters() if p.requires_grad),
                  lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"] * len(train_loader),
                              eta_min=CONFIG["scheduler_eta_min"])


def run_epoch(epoch):
    model.train()
    agg = {"total": 0.0, "ce": 0.0, "dcs": 0.0, "n": 0}
    for step, (img_clean, img_deg, labels) in enumerate(train_loader):
        img_clean = img_clean.to(DEVICE, non_blocking=True)
        img_deg = img_deg.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            total, ce, dcs = model.compute_loss(img_clean, img_deg, labels)
        total.backward()
        torch.nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad), CONFIG["max_grad_norm"])
        optimizer.step()
        scheduler.step()
        bs = labels.size(0)
        agg["total"] += float(total.detach()) * bs
        agg["ce"] += float(ce.detach()) * bs
        agg["dcs"] += float(dcs.detach()) * bs
        agg["n"] += bs
        if step % 20 == 0:
            logger.info(f"ep{epoch} step {step}/{len(train_loader)} | "
                        f"total {agg['total']/max(agg['n'],1):.4f} (ce {agg['ce']/max(agg['n'],1):.4f}, "
                        f"dcs {agg['dcs']/max(agg['n'],1):.4f}) | lr {optimizer.param_groups[0]['lr']:.2e}")
    return {k: agg[k] / max(agg["n"], 1) for k in ("total", "ce", "dcs")}


@torch.no_grad()
def run_val():
    model.eval()
    agg = {"total": 0.0, "n": 0}
    scores, labels_all = [], []
    for img_clean, img_deg, labels in val_loader:
        img_clean = img_clean.to(DEVICE, non_blocking=True)
        img_deg = img_deg.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            total, _, _ = model.compute_loss(img_clean, img_deg, labels)
            out = model(img_clean)
        logits = out["logits"].float()
        scores.append((logits[:, 1] - logits[:, 0]).cpu())
        labels_all.append(labels.cpu())
        agg["total"] += float(total) * labels.size(0)
        agg["n"] += labels.size(0)
    scores = torch.cat(scores).numpy()
    labels_all = torch.cat(labels_all).numpy()
    return {
        "val_loss": agg["total"] / max(agg["n"], 1),
        "val_auc": roc_auc_score(labels_all, scores),
        "val_bacc": balanced_accuracy_score(labels_all, scores > 0),
    }

LOG_CSV = OUTPUTS / "train_log.csv"
with open(LOG_CSV, "w", encoding="utf-8") as f:
    f.write("epoch,train_loss,train_ce,train_dcs,val_loss,val_auc,val_bacc\n")

TRAIN_LOG = []
best_bacc = 0.0
for epoch in range(1, CONFIG["epochs"] + 1):
    tr = run_epoch(epoch)
    va = run_val()
    logger.info(f"--> epoch {epoch}: train total={tr['total']:.4f} (ce={tr['ce']:.4f}, dcs={tr['dcs']:.4f}) | "
                f"val loss={va['val_loss']:.4f} auc={va['val_auc']:.4f} bacc={va['val_bacc']:.4f}")
    TRAIN_LOG.append({"epoch": epoch, "train_loss": tr["total"], "train_ce": tr["ce"],
                      "train_dcs": tr["dcs"], **va})
    with open(LOG_CSV, "a", encoding="utf-8") as f:
        f.write(f"{epoch},{tr['total']:.4f},{tr['ce']:.4f},{tr['dcs']:.4f},"
                f"{va['val_loss']:.4f},{va['val_auc']:.4f},{va['val_bacc']:.4f}\n")
    if va["val_bacc"] > best_bacc:
        best_bacc = va["val_bacc"]
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_bacc": best_bacc,
                    "config": CONFIG},
                   OUTPUTS / "bfree_globalforge_lora_r16_best.pth")
        logger.info(f"[*] best checkpoint saved (epoch {epoch}, bAcc {best_bacc:.4f})")

torch.save({"model": model.state_dict(), "epoch": CONFIG["epochs"], "config": CONFIG,
            "train_log": TRAIN_LOG},
           OUTPUTS / "bfree_globalforge_lora_r16.pth")
logger.info("TRAINING COMPLETE")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(LOG_CSV)
display(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=120)
axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="total")
axes[0].plot(df["epoch"], df["train_ce"], marker="s", label="CE")
axes[0].plot(df["epoch"], df["train_dcs"], marker="^", label="DCS")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train loss"); axes[0].legend()
axes[0].set_title("Train losses (CE & DCS logged separately)")
axes[1].plot(df["epoch"], df["val_loss"], marker="o", color="tab:red")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val loss"); axes[1].set_title("Val loss")
axes[2].plot(df["epoch"], df["val_auc"], marker="o", label="AUC")
axes[2].plot(df["epoch"], df["val_bacc"], marker="s", label="bAcc")
axes[2].set_xlabel("epoch"); axes[2].set_title("Val metrics"); axes[2].legend()
fig.tight_layout()
fig.savefig(OUTPUTS / "training_curves.png")
logger.info("saved training_curves.png")

## Training Complete (K2)

Outputs trong `/kaggle/working/`:
- `bfree_globalforge_lora_r16.pth` / `..._best.pth` — checkpoint cuối / best-val_bAcc (kèm CONFIG — K3 cần để tái tạo kiến trúc)
- `train_log.csv` — CE/DCS riêng từng epoch (cho K4 / Phase 8)
- `training_curves.png` · `train_log.txt` — log đầy đủ

**K2 checklist:** chạy hết 8 epochs không lỗi · checkpoint saved · training log CSV + curves.

**Bước giao tiếp:** **Save Version → Save & Run All (Commit)**, rồi mở notebook 03 → *Add Input → Your Work → notebook 02 này*.